# Stage 2: the baselines

**The question.** Before looking at what a video says or shows, how well can its lift be predicted from who made
it and when? `lift` is stage 1's target: the video's log share rate minus the median of its creator's last 30
videos posted at least 14 days earlier (notebook 01, cell 27). Four baselines, each simpler than any content model:

| | Baseline | Predicts lift as |
|---|---|---|
| B0 | Zero lift | 0: every video does as well as its creator's recent median |
| B1 | Recent form | the median of the creator's last 5 or 10 settled videos, or an exponentially weighted mean of them, minus the baseline |
| B2 | The creator's drift | the mean lift of the creator's earlier videos, shrunk toward 0 by n / (n + k) |
| B3 | Metadata | ridge and gradient-boosted trees on duration, posting hour and weekday, days since the previous video, the number of earlier videos, and wordless |

Every later stage has to beat the best of these. The code is in `src/sharerate` (the loader, the history features,
the metrics and the bootstrap), with its tests in `tests/`.

## 0. The bar, written before the first run (rule 7)

This cell was written, and committed, before any baseline was scored.

- **The label** is stage 1's as-of lift (`targets.lift`). Every history feature follows the same as-of rule: only
  videos posted strictly earlier, and for anything built from their share rates, only videos posted at least 14
  days earlier.
- **The rows** are the validation rows of `targets`, 21,332 videos from 203 creators, the same for every baseline.
  A baseline that needs more history than a row has predicts zero lift on that row, and the count is stated.
- **The primary bar.** A baseline beats zero lift if the 95% paired creator-bootstrap interval of its validation
  MAE gain (B0's MAE minus its own) lies above 0 for **seen** creators: 138 creators, 13,646 videos. New creators
  (27, 2,275 videos) and held-out creators (38, 5,411) are reported alongside and are not required to pass.
- **The secondary bar.** The gain in breakout ROC-AUC over B0's 0.5 has an interval above 0, and so does the gain
  in the top-10% share over B0's. B0's top-10% share is each creator's own breakout rate, close to the 0.242 base
  rate, so the paired gain compares a model with exactly what a random pick would get.
- **A miss is a miss.** A gain whose interval touches 0 is reported as "no better than the creator's recent
  median", never tuned until it passes. That result is publishable.
- **Every setting is chosen on validation,** by MAE over all 21,332 rows: B1's half-life, B2's k, the ridge
  penalty, the trees' size and number. The test split is never opened here; stage 5 opens it once.
- **Intervals** come from 2,000 replicates that resample creators, not videos, with every model and B0 scored on
  the same replicates.